# IT2011 Final Evaluation - Simple Linear Regression
**Student:** IT25102520

**Problem:** Predict the average temperature (TAVG) and the rainfall (PRCP_log) for Warsaw.

**Libraries:** pandas, numpy, matplotlib, scikit-learn.

**Implementation:** LinearRegression with a single input feature, no scaling, `train_test_split(test_size=0.2, random_state=42)` (same split as the whole group).

**Tuning:** Simple linear regression has no hyperparameters, so each candidate feature was tested on its own using 5-fold cross-validation on the training set only. The feature with the highest mean R2 was selected.

## Part 1: Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

In [ ]:
# path to the processed dataset (notebook is inside the notebooks folder)
# change the file name here if the group saved it with a different name
DATA_PATH = "../results/outputs/warsaw_processed.csv"

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

In [ ]:
# check missing values and basic statistics
print(df.isna().sum())
df.describe()

In [ ]:
# tests each feature alone and ranks them by mean 5-fold CV R2
def test_single_features(X_train, y_train):
    results = []
    for col in X_train.columns:
        score = cross_val_score(LinearRegression(), X_train[[col]], y_train,
                                cv=5, scoring="r2").mean()
        results.append((col, score))
    table = pd.DataFrame(results, columns=["Feature", "CV_R2"])
    table = table.sort_values("CV_R2", ascending=False).reset_index(drop=True)
    return table

## Part 2: Task A - Predict temperature (TAVG)

In [ ]:
season_cols = [col for col in df.columns if col.startswith("Season_")]

y_a = df["TAVG"]
X_a = df[["SNWD", "TMIN", "PRCP_log"] + season_cols]   # TMAX and DATE left out

Xa_train, Xa_test, ya_train, ya_test = train_test_split(
    X_a, y_a, test_size=0.2, random_state=42)

In [ ]:
res_a = test_single_features(Xa_train, ya_train)
res_a

In [ ]:
best_a = res_a.loc[0, "Feature"]
print("Best single feature for TAVG:", best_a)

model_a = LinearRegression()
model_a.fit(Xa_train[[best_a]], ya_train)
pred_a = model_a.predict(Xa_test[[best_a]])

r2_a = r2_score(ya_test, pred_a)
rmse_a = np.sqrt(mean_squared_error(ya_test, pred_a))

print("Intercept:", round(model_a.intercept_, 4), "| Slope:", round(model_a.coef_[0], 4))
print("R2   :", round(r2_a, 4))
print("RMSE :", round(rmse_a, 4))

In [ ]:
order = np.argsort(Xa_test[best_a].values)
plt.figure(figsize=(7, 5))
plt.scatter(Xa_test[best_a], ya_test, alpha=0.4, label="Actual")
plt.plot(Xa_test[best_a].values[order], pred_a[order], color="red", label="Regression line")
plt.xlabel(best_a)
plt.ylabel("TAVG")
plt.title("Simple Linear Regression: TAVG vs " + best_a)
plt.legend()
plt.show()

## Part 3: Task B - Predict rainfall (PRCP_log)

In [ ]:
y_b = df["PRCP_log"]
X_b = df[["SNWD", "TAVG", "TMAX", "TMIN"] + season_cols]

Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    X_b, y_b, test_size=0.2, random_state=42)

In [ ]:
res_b = test_single_features(Xb_train, yb_train)
res_b

In [ ]:
best_b = res_b.loc[0, "Feature"]
print("Best single feature for PRCP_log:", best_b)

model_b = LinearRegression()
model_b.fit(Xb_train[[best_b]], yb_train)
pred_b = model_b.predict(Xb_test[[best_b]])

r2_b = r2_score(yb_test, pred_b)
rmse_b = np.sqrt(mean_squared_error(yb_test, pred_b))

print("Intercept:", round(model_b.intercept_, 4), "| Slope:", round(model_b.coef_[0], 4))
print("R2   :", round(r2_b, 4))
print("RMSE :", round(rmse_b, 4))

In [ ]:
order = np.argsort(Xb_test[best_b].values)
plt.figure(figsize=(7, 5))
plt.scatter(Xb_test[best_b], yb_test, alpha=0.4, label="Actual")
plt.plot(Xb_test[best_b].values[order], pred_b[order], color="red", label="Regression line")
plt.xlabel(best_b)
plt.ylabel("PRCP_log")
plt.title("Simple Linear Regression: PRCP_log vs " + best_b)
plt.legend()
plt.show()

## Part 4: Final summary

In [ ]:
summary = pd.DataFrame({
    "Task": ["Temperature (TAVG)", "Rainfall (PRCP_log)"],
    "Model": ["Simple Linear Regression"] * 2,
    "Feature used": [best_a, best_b],
    "R2": [round(r2_a, 4), round(r2_b, 4)],
    "RMSE": [round(rmse_a, 4), round(rmse_b, 4)],
})
summary

## Discussion
*(Write this in your own words after you see your results. Points to cover: which feature was best for each task and why, how well a one-feature linear model explains temperature compared with rainfall, and the limitations, such as the random split on time-ordered data and the fact that rainfall is non-linear.)*